# 🍜 Nhận Diện Món Ăn Việt Nam bằng CNN

**Pipeline:** Mount Drive → Copy data local → Augmentation → CNN → Train → Predict

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Import thư viện

In [ ]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 3. Cấu hình

> ⚙️ Chỉ cần đổi `DRIVE_DATA_DIR` cho đúng với Drive của bạn.

In [ ]:
# ============================================================
# ⚙️  CHỈ CẦN ĐỔI DÒNG NÀY cho đúng đường dẫn Drive của bạn
# ============================================================
DRIVE_DATA_DIR = '/content/drive/MyDrive/ai/images'

# Các tham số huấn luyện
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 60
SEED       = 42

## 4. Copy data từ Drive về local

> Bước này giúp train **nhanh hơn 20-50x** so với đọc thẳng từ Drive.
> Chỉ mất 1-3 phút, chạy 1 lần duy nhất.

In [ ]:
LOCAL_DATA_DIR = '/content/images'

if os.path.exists(LOCAL_DATA_DIR):
    print(f'Đã có sẵn tại {LOCAL_DATA_DIR}, bỏ qua copy.')
else:
    print('Đang copy từ Drive về local...')
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    print('Copy xong!')

# Kiểm tra
CLASS_NAMES = sorted([
    d for d in os.listdir(LOCAL_DATA_DIR)
    if os.path.isdir(os.path.join(LOCAL_DATA_DIR, d))
])
NUM_CLASSES = len(CLASS_NAMES)

print(f'\nTìm thấy {NUM_CLASSES} lớp:')
for i, c in enumerate(CLASS_NAMES):
    n = len(os.listdir(os.path.join(LOCAL_DATA_DIR, c)))
    print(f'  [{i:2d}] {c:25s} -> {n} ảnh')

## 5. Data Augmentation & DataLoader

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    channel_shift_range=20,
    fill_mode='nearest'
)

train_gen = train_datagen.flow_from_directory(
    LOCAL_DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=SEED
)

IDX2CLASS = {v: k for k, v in train_gen.class_indices.items()}
print('Thứ tự class:', train_gen.class_indices)

### Xem thử ảnh sau augmentation

In [ ]:
imgs, lbls = next(train_gen)
fig, axes = plt.subplots(3, 6, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i])
    ax.set_title(IDX2CLASS[np.argmax(lbls[i])], fontsize=8)
    ax.axis('off')
plt.suptitle('Ảnh train sau augmentation', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Xây dựng mô hình CNN

In [ ]:
def build_cnn(input_shape, num_classes):
    model = models.Sequential([
        # Khối 1 - 32 filters
        layers.Conv2D(32, (3,3), padding='same', input_shape=input_shape),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.Conv2D(32, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D(2, 2), layers.Dropout(0.25),

        # Khối 2 - 64 filters
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D(2, 2), layers.Dropout(0.25),

        # Khối 3 - 128 filters
        layers.Conv2D(128, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.Conv2D(128, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D(2, 2), layers.Dropout(0.30),

        # Khối 4 - 256 filters
        layers.Conv2D(256, (3,3), padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D(2, 2), layers.Dropout(0.30),

        # Fully Connected
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_cnn((*IMG_SIZE, 3), NUM_CLASSES)
model.summary()

## 7. Compile & Callbacks

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

CKPT_PATH = '/content/best_food_cnn.keras'

callbacks = [
    ReduceLROnPlateau(monitor='loss', factor=0.5,
                      patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint(CKPT_PATH, monitor='accuracy',
                    save_best_only=True, verbose=1)
]
print('Sẵn sàng train!')

## 8. Huấn luyện

In [ ]:
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

## 9. Đồ thị Loss & Accuracy

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['accuracy'], linewidth=2, color='#2ecc71')
ax1.set_title('Train Accuracy')
ax1.set_xlabel('Epoch')
ax1.grid(alpha=0.3)

ax2.plot(history.history['loss'], linewidth=2, color='#e74c3c')
ax2.set_title('Train Loss')
ax2.set_xlabel('Epoch')
ax2.grid(alpha=0.3)

plt.suptitle('Quá trình huấn luyện CNN', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Lưu model về Drive

In [ ]:
best_model = keras.models.load_model(CKPT_PATH)

SAVE_PATH = '/content/drive/MyDrive/ai/food_cnn_model.keras'
best_model.save(SAVE_PATH)
print(f'✅ Model đã lưu tại: {SAVE_PATH}')

## 11. Dự đoán ảnh — Upload từ máy

> Chạy cell này bất cứ lúc nào muốn test, upload được nhiều ảnh cùng lúc.

In [ ]:
from google.colab import files

def predict_image(img_path, model, idx2class, top_k=3):
    img     = keras_image.load_img(img_path, target_size=IMG_SIZE)
    arr     = keras_image.img_to_array(img) / 255.0
    arr     = np.expand_dims(arr, axis=0)
    preds   = model.predict(arr, verbose=0)[0]
    top_idx = np.argsort(preds)[::-1][:top_k]

    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 4))
    ax_img.imshow(img)
    ax_img.set_title(os.path.basename(img_path), fontsize=10)
    ax_img.axis('off')

    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(top_k)]
    bars = ax_bar.barh(
        [idx2class[i] for i in top_idx[::-1]],
        [preds[i]*100 for i in top_idx[::-1]],
        color=colors[::-1]
    )
    ax_bar.set_xlabel('Xác suất (%)')
    ax_bar.set_xlim(0, 100)
    ax_bar.set_title(f'Top-{top_k} dự đoán')
    for bar, idx in zip(bars[::-1], top_idx):
        ax_bar.text(bar.get_width() + 0.5,
                    bar.get_y() + bar.get_height() / 2,
                    f'{preds[idx]*100:.1f}%', va='center', fontsize=10)

    plt.suptitle(
        f'Kết quả: {idx2class[top_idx[0]]}  ({preds[top_idx[0]]*100:.1f}%)',
        fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

uploaded = files.upload()
for fname in uploaded:
    predict_image(fname, best_model, IDX2CLASS)